# Transferring Topo stuff to python

In [ ]:
# Import packages 
import os
import sys
workdir_ = '/glade/work/juliob'
if ( workdir_ not in sys.path ):
    sys.path.append(workdir_)
    print( f" a path to {workdir_} added in {__name__} ")

import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature




from myPythonTools.TopoAna import Plots as Po
from myPythonTools.TopoAna import rdtilemap as Ti
from myPythonTools.TopoAna import rdremap as RM
from myPythonTools.TopoAna import rdridgelist as RL


import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.tri as tri
import matplotlib.colors as colors

from scipy.io import FortranFile

import importlib
import glob
import copy

importlib.reload(Po)
importlib.reload(Ti)
importlib.reload(RM)
importlib.reload(RL)

#sys.path.append('../Plotting/')
#   Now you can imprt modules in ../Plotting
#import xyp_plot as xyp
#import ana as a


In [ ]:
host=os.getenv('HOST')
host

In [ ]:
# ============================================================
# 6. Helper function for one panel
# ============================================================

def plot_ridgelet_panel(ax, fill_field, panel_title, ridgelet_length_fac=1.0, ridgelet_color_override=None, plot_ridgelets=True, alpha=0.5 ):
    """
    fill_field should be either raw or smoo
    """

    # Filled topography
    cf = ax.contourf(
        tlons[ip, yslice, xslice],
        tlats[ip, yslice, xslice],
        fill_field[ip, yslice, xslice],
        levels=topo_levels,
        cmap=cmap,
        norm=norm,
        alpha=alpha,
        transform=DataProj
    )

    # Filled topography
    cf = ax.contour(
        tlons[ip, yslice, xslice],
        tlats[ip, yslice, xslice],
        raw[ip, yslice, xslice],
        levels=[0.1,10_000.],
        colors='black',
        norm=norm,
        alpha=0.8,
        transform=DataProj
    )
    raw_plot = raw[ip, yslice, xslice]
    
    ax.contourf(
        tlons[ip, yslice, xslice],
        tlats[ip, yslice, xslice],
        raw_plot,
        levels=[np.nanmin(raw_plot) - 1., 0.1],
        colors=['lightskyblue'],
        alpha=0.8,
        transform=DataProj,
        zorder=5
    )
    plot_lngth = ridgelet_length_fac * lngth
    
    # Ridgelets colored by mxdis + local smoothed target_topo
    if plot_ridgelets == True:
        for i in iroo:
            for ir in range(maxridges):
    
                value = mxdis[i, ir] 
    
                if not np.all(np.isfinite([
                    latc[i, ir],
                    lonc[i, ir],
                    angll[i, ir],
                    lngth[i, ir],
                    value
                ])):
                    continue
                    
                
                lat0 = latc[i, ir]
                lon0 = lonc[i, ir]
    
                lat1, lon1, lat2, lon2 = Po.CAMridgelet(
                    lat=lat0,
                    lon=lon0,
                    angll=angll[i, ir],
                    clngt=plot_lngth[i, ir]
                )
    
                if ridgelet_color_override is None:
                    ridge_color = cmap(norm(value))
                else:
                    ridge_color = ridgelet_color_override
                    
                ax.plot(
                    [lon1, lon2],
                    [lat1, lat2],
                    color=ridge_color,
                    linewidth=ridge_linewidth,
                    transform=DataProj,
                    zorder=10
                )
    
    ax.set_extent(
        [lonrange[0], lonrange[1], latrange[0], latrange[1]],
        crs=DataProj
    )

    gl = ax.gridlines(
        draw_labels=True,
        linewidth=1,
        color='gray',
        alpha=0.5,
        linestyle='--'
    )

    gl.top_labels   = False
    gl.right_labels = False
    gl.xlabel_style = {'size': 11, 'color': 'gray'}
    gl.ylabel_style = {'size': 11, 'color': 'gray'}

    ax.set_title(panel_title, fontsize=13)

    return cf





## Set some paths and directories

In [ ]:
TopoRoot = '/glade/work/juliob/Topo/NCARTopoJTB-fire/'     #cases/ne30pg3_Sco100_small_rdgs/

# Note only cases with v202 and higher, e.g., v203 (or created after 9/7/2026)
# have good clngt in *tiles.nc file.

Case = 'ne30pg3_Sco100_Rw050_v202_rch1p0_e01'
Case = 'ne30pg3_Sco100_Rw050_v203_xtra_e02'
Case = 'ne30pg3_Sco100_Rw050_v205_new_wedge_e03'
ogrid= 'ne30pg3'



In [ ]:
import glob
import os
import xarray as xr
import numpy as np

outdir = TopoRoot + '/cases/' + Case + '/output/'

# Find ordinary files, explicitly excluding *_tiles.nc
foo = [
    f for f in glob.glob(outdir + ogrid + '*.nc')
    if not f.endswith('_tiles.nc')
]

# Take newest file
FileN = max(foo, key=os.path.getmtime)

# Corresponding tiles file
FileTiles = FileN.replace('.nc', '_tiles.nc')

if not os.path.exists(FileTiles):
    raise FileNotFoundError(f"No corresponding tiles file:\n{FileTiles}")

print("Main file: ", FileN)
print("Tiles file:", FileTiles)

# Main grid file
dCAM = xr.open_dataset(FileN)

print(list(dCAM.variables))
print(np.shape(dCAM['lat']))

target_lon  = dCAM['lon'].values
target_lat  = dCAM['lat'].values
target_topo = dCAM['PHIS'].values / 9.8

# Tile file
dTiles = xr.open_dataset(FileTiles)

print(list(dTiles.variables))

In [ ]:
clng1=dCAM.CLNGT.values
clng2=dTiles.CLNGT.values

plt.scatter( np.sum(clng1,axis=0)  , np.sum(clng2,axis=0)    )
plt.title( Case )
#plt.plot( np.sum(clng2,axis=0) )


In [ ]:

# Do a file search in output directory
# Pick last (most recent) file
foo=glob.glob( TopoRoot +'/cases/'+ Case + '/output/Ridge_list*dat')
ListFile = foo[-1]
List=RL.rdridgelist( ListFile )

foo=glob.glob( TopoRoot +'/cases/'+ Case + '/output/remap*dat')
RmpFile = foo[-1]
print(RmpFile)
Rmp=RM.rdremap( RmpFile,  fields="all" )

foo=glob.glob( TopoRoot +'/cases/'+ Case + '/output/topo*.nc')
FileN = foo[-1]
print(FileN)
topo=xr.open_dataset( FileN )
print(list(topo.variables))
print(topo.attrs)
raw=topo['terr_dev']+topo['terr_sm']
Loo = np.size(raw)
nc=int(np.sqrt( Loo/6 ))
print(int(nc))
raw=np.reshape( raw.values , (6,nc,nc) )
dev=np.reshape( topo['terr_dev'].values , (6,nc,nc) )
smoo=np.reshape( topo['terr_sm'].values , (6,nc,nc) )
tlats=np.reshape( topo['lat'].values , (6,nc,nc) )
tlons=np.reshape( topo['lon'].values , (6,nc,nc) )

# Define the range of indices
x_indices = np.arange(3000)
y_indices = np.arange(3000)

# Use meshgrid to create 2D arrays of x and y indices
xgrid, ygrid = np.meshgrid(x_indices, y_indices, indexing='xy')


foo=glob.glob( TopoRoot +'/cases/'+ Case + '/output/Ridge_tile_map.dat')
TileFile = foo[-1]
Tile=Ti.rdtilemap( tile=TileFile, read_angll=True )

ntarget,maxridges = Tile.MyObject.shape
print(ntarget,maxridges)
print( f"Tile shape {Tile.lonc.shape}" )


latc=(180./np.pi)*Tile.latc
lonc=(180./np.pi)*Tile.lonc
angll=Tile.angll
anglx=Tile.anglx
aniso=Tile.aniso
lngthvar = 'LnObject'
lngth=Tile[ lngthvar ] # 
#clngt=Tile.clngt
mxdis=Tile.mxdis



In [ ]:
import rdTerrXY as Te
importlib.reload(Te)

case_dir=TopoRoot +'/cases/'+ Case 
Terr = Te.rdterrxy(Te.latest_terrxy(case_dir), verbose=True)

In [ ]:
print( Terr.keys() )
print( Terr.ipk.shape )
plt.plot( Terr.ipk )

In [ ]:
latlim=np.asarray( [34.5,36.] )
lonlim=360.- np.asarray( [85,82.5] )
xslim = np.asarray([1670,1760])
yslim = np.asarray([2654,2705])
Panel = 4



latc=(180./np.pi)*List.latc
lonc=(180./np.pi)*List.lonc
inbox = ((latc >= latlim.min()) & (latc <= latlim.max()) &
         (lonc >= lonlim.min()) & (lonc <= lonlim.max()) &
         (List.latc > -900))                      # drop the -9999 'no wedge' entries

xspk =List.xspk
yspk =List.yspk
ipan =List.MyPanel
"""
inbox = ((yspk >= yslim.min()) & (yspk <= yslim.max()) &
         (xspk >= xslim.min()) & (xspk <= xslim.max()) &
         (ipan == Panel))                      # drop the -9999 'no wedge' entries
"""

idx = np.flatnonzero(inbox)                       # peak indices (0-based, = ipk-1)
print(len(idx), 'ridges in box')
#print( ipan[idx] )


In [ ]:
print(list(dTiles.variables))

In [ ]:
plt.plot(dTiles.MXDIS.values.flatten() )

In [ ]:
lngth2=dTiles.CLNGT.values
print(lngth.shape)
print(lngth2.shape)
plt.scatter(  lngth2.T.flatten()  ,  lngth.flatten()  )

In [ ]:


latc2=(180./np.pi)*dTiles.latc.values.T
lonc2=(180./np.pi)*dTiles.lonc.values.T

inbox2 = ((latc2 >= latlim.min()) & (latc2 <= latlim.max()) &
         (lonc2 >= lonlim.min()) & (lonc2 <= lonlim.max()) )     

idx2 = np.flatnonzero(inbox2)                       # peak indices (0-based, = ipk-1)
print(len(idx2), 'ridges in box')
#print( ipan[idx] )


In [ ]:
print(idx2)

In [ ]:
fig,axs=plt.subplots( 1,2, figsize=(2*8,7) )
axs[0].scatter( lonc[idx],latc[idx] )
axs[1].scatter( lonc2.flatten()[idx2],latc2.flatten()[idx2] )
#axs[1].scatter( xspk[idx],yspk[idx] )
#plt.xlim(0,360)
#plt.ylim(-90,90)

In [ ]:
print( stop_here )

In [ ]:
print(List.keys())
print(Rmp.keys())


In [ ]:
k_sierra  = Terr.index_of(265570)
k_sangab  = Terr.index_of(262660)
k_smokies = Terr.index_of(260233)

print( k_sierra , k_sangab, k_smokies )


In [ ]:
plt.plot( List.yspk )

In [ ]:
List.keys()

In [ ]:
import numpy as np, matplotlib
#matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch

def north_arrow(ax, frac=(0.88, 0.72), length=0.18, label="N", color="k", fs=16):
    """Vertical arrow in axes coords. On cube panel 4, +y is exactly north."""
    x, y = frac
    ax.annotate("", xy=(x, y+length), xytext=(x, y), xycoords="axes fraction",
                arrowprops=dict(arrowstyle="-|>", lw=2.5, color=color))
    ax.text(x, y+length+0.02, label, transform=ax.transAxes, ha="center",
            va="bottom", fontsize=fs, color=color, fontweight="bold")

def dir_arrow(ax, frac=(0.62, 0.08), length=0.22, label="cross-ridge",
              color="k", fs=13, horizontal=True):
    """Horizontal arrow in axes coords, for the cross-ridge direction."""
    x, y = frac
    dx, dy = (length, 0) if horizontal else (0, length)
    ax.annotate("", xy=(x+dx, y+dy), xytext=(x, y), xycoords="axes fraction",
                arrowprops=dict(arrowstyle="-|>", lw=2.5, color=color))
    ax.text(x+dx/2, y+dy+0.03, label, transform=ax.transAxes, ha="center",
            va="bottom", fontsize=fs, color=color)

def rotation_arrow(ax, angle_deg, frac=(0.5, 0.5), radius=0.30, rad=0.35,
                   color="crimson", lw=3, fs=15, sign=+1, label=None):
    """Curved arrow showing rotation of the field through angle_deg.

    `sign` flips the sweep direction. rotby4's convention is easy to get
    backwards, so look at the figure and flip if it disagrees with the panels.
    """
    cx, cy = frac
    a0 = np.radians(90.0)                       # start pointing up (north)
    a1 = np.radians(90.0 + sign*angle_deg)      # end at the crest direction
    p0 = (cx + radius*np.cos(a0), cy + radius*np.sin(a0))
    p1 = (cx + radius*np.cos(a1), cy + radius*np.sin(a1))
    ax.add_patch(FancyArrowPatch(
        p0, p1, transform=ax.transAxes, clip_on=False,
        connectionstyle=f"arc3,rad={sign*rad}",
        arrowstyle="-|>", mutation_scale=28, lw=lw, color=color))
    if label is None:
        label = f"rotate {angle_deg:.0f}$^\\circ$"
    am = 0.5*(a0+a1)
    ax.text(cx + 1.35*radius*np.cos(am), cy + 1.35*radius*np.sin(am), label,
            transform=ax.transAxes, ha="center", va="center",
            fontsize=fs, color=color)


def north_arrow_rotated(ax, anglx_deg, frac=(0.80, 0.62), length=0.20,
                        flip=False, color="k", fs=15, label="N", aspect=1.0):
    """North arrow in the ROTATED frame.

    The rotation is rigid, so north's direction follows from the crest alone.
    The crest starts at math angle (90 - anglx) -- anglx is azimuth, clockwise
    from north -- and ends vertical, so the field was rotated by +anglx.
    North therefore moves from 90 to (90 + anglx).

    A ridge line is undirected, so the crest may end up pointing up or down;
    that is a 180 degree ambiguity, not a sign change. Set flip=True if the
    arrow disagrees with the unrotated panel.

    Check: the angle between this arrow and vertical equals anglx (mod 180),
    and north sits on the same side of the crest as in the unrotated panel.

    aspect: pass (axes height / axes width) in display units if the panel is
    not square, or just call ax.set_aspect('equal').
    """
    th = np.radians(90.0 + anglx_deg + (180.0 if flip else 0.0))
    dx, dy = length*np.cos(th), length*np.sin(th)*aspect
    x, y = frac
    ax.annotate("", xy=(x+dx, y+dy), xytext=(x, y), xycoords="axes fraction",
                arrowprops=dict(arrowstyle="-|>", lw=2.5, color=color))
    ax.text(x+1.30*dx, y+1.30*dy, label, transform=ax.transAxes,
            ha="center", va="center", fontsize=fs, color=color, fontweight="bold")
    return (np.degrees(th)) % 360


"""
# --- demo ---
fig, axs = plt.subplots(1, 4, figsize=(22, 5))
anglx = 118.0
for a in axs:
    a.set_xticks([]); a.set_yticks([])
axs[0].set_title("terrain, cube frame"); north_arrow(axs[0])
axs[1].set_title("analysis window");     north_arrow(axs[1])
axs[2].set_title("rotated to crest")
rotation_arrow(axs[2], anglx, frac=(0.5, 0.52), sign=-1)
dir_arrow(axs[2], label="cross-ridge")
axs[3].set_title("cross-ridge profile"); dir_arrow(axs[3], frac=(0.55, 0.06))
fig.savefig("arrows_demo.png", dpi=80, bbox_inches="tight")
print("ok - arrows_demo.png written")
"""

In [ ]:

GLS = 6371.0*np.pi/2/3000.0        # 3.3358 km per cube cell at ncube=3000

def km_ticks(ax, x0=None, y0=None, gls=GLS, step=25, axis="both",
             label="distance (km)", fmt="{:.0f}"):
    """Relabel index axes as km measured from a centre, leaving the data alone.

    x0, y0 : index of the origin. For a rotated window use Terr.PSW; for the
             raw terrain panel use the peak location (xspk, yspk).
    step   : tick spacing in km.

    Call AFTER set_xlim / set_ylim, since the tick positions are chosen to
    cover the current limits.
    """
    if axis in ("both", "x") and x0 is not None:
        lo, hi = ax.get_xlim()
        kms = np.arange(np.ceil((lo-x0)*gls/step)*step,
                        (hi-x0)*gls + 1e-9, step)
        ax.set_xticks(x0 + kms/gls)
        ax.set_xticklabels([fmt.format(k) for k in kms])
        ax.set_xlabel(label)
    if axis in ("both", "y") and y0 is not None:
        lo, hi = ax.get_ylim()
        kms = np.arange(np.ceil((lo-y0)*gls/step)*step,
                        (hi-y0)*gls + 1e-9, step)
        ax.set_yticks(y0 + kms/gls)
        ax.set_yticklabels([fmt.format(k) for k in kms])
        ax.set_ylabel(label)
        
def panel_label(ax, text, fontsize=26, pad=0.02):
    return ax.text(pad, 1.0-pad, text, transform=ax.transAxes,
                   fontsize=fontsize, fontweight='bold',
                   va='top', ha='left', zorder=20,
                   bbox=dict(facecolor='white', edgecolor='none',
                             alpha=0.75, pad=4))

def annotate_wedge(ax, xw, hw, gls=GLS, color='darkorange', fs=15,
                   guides=True, hside='right', lw=2.5):
    """Mark h (max vertical span) and W (max horizontal span) of the wedge.

    xw, hw : the three wedge nodes in the SAME coordinates used to plot them,
             i.e. xw already offset (18 + List.xwedge_list[ipk-1,:]).
    Returns (h_metres, W_km).
    """
    xw = np.asarray(xw, float); hw = np.asarray(hw, float)
    htop, hbot = hw.max(), hw.min()
    h_m  = htop - hbot
    W_km = (xw.max() - xw.min()) * gls

    xpk = xw[int(np.argmax(hw))]
    dx  = 0.9 if hside == 'right' else -0.9
    xh  = xpk + dx

    if guides:
        for y in (htop, hbot):
            ax.plot([min(xw.min(), xh), max(xw.max(), xh)], [y, y],
                    ls=':', lw=1.2, color=color, zorder=3)

    ax.annotate("", xy=(xh, htop), xytext=(xh, hbot),
                arrowprops=dict(arrowstyle="<->", lw=lw, color=color), zorder=6)
    ax.text(xh + 0.35*np.sign(dx), 0.5*(htop+hbot), f"h = {h_m:.0f} m",
            color=color, fontsize=fs, va='center',
            ha='left' if dx > 0 else 'right', zorder=7)

    yspan = ax.get_ylim()[1] - ax.get_ylim()[0]
    yW = hbot - 0.03*yspan
    ax.annotate("", xy=(xw.max(), yW), xytext=(xw.min(), yW),
                arrowprops=dict(arrowstyle="<->", lw=lw, color=color), zorder=6)
    ax.text(0.5*(xw.min()+xw.max()), yW - 0.02*yspan, f"W = {W_km:.0f} km",
            color=color, fontsize=fs, ha='center', va='top', zorder=7)
    return h_m, W_km


In [ ]:
%matplotlib inline
#k_smokies = Terr.index_of(260233)

k=k_sierra #Terr.index_of(230_401) #k_smokies # k_smokies #k_sangab

rtk=Terr.rt[k]
#rtk=np.where( rtk < -10000., 0., rtk )
print( rtk.shape )



ipk = Terr.ipk[k]
print(List.mxdis[ipk-1], List.anglx[ipk-1], List.clngth[ipk-1],List.NSW,List.PSW)

xedge=List

print( List.hwedge_list[ipk-1,:] )
print( 17+List.xwedge_list[ipk-1,:] )

print( Terr.xspk[k],Terr.yspk[k])
xspk,yspk= Terr.xspk[k],Terr.yspk[k]




plt.rcParams.update({
    'axes.labelsize':  18,
    'xtick.labelsize': 15,
    'ytick.labelsize': 15,
    'axes.titlesize':  18,
})
# plt.rcdefaults()

fig,axs=plt.subplots( 1,3, figsize=(31,9) )

i0, j0 = int(round(xspk)), int(round(yspk))
maxraw = np.max(dev[3, j0-30:j0+30, i0-30:i0+30])

#topolevs=np.linspace( 0,maxraw,num=51)

topolevs = np.linspace(-1000., 1000., 21) # good for smokies
topolevs = 2*np.linspace(-1000., 1000., 21) # good for smokies

ax=axs[0]
#ax.contour(Rmp.mxdis[3,:,:] )
ax.contourf(dev[3,:,:] , cmap='terrain', levels=topolevs)
ax.contour(Rmp.mxdis[3,:,:] , colors='black')
ax.set_xlim(xspk-30,xspk+30) # 300,800 )
ax.set_ylim(yspk-30,yspk+30) #2500,3000 )
ax.plot( xspk,yspk , '+'  )
km_ticks(ax, x0=xspk,      y0=yspk,      step=50)

"""
ax=axs[1]
ax.contourf( Terr.suba[k].T , cmap='terrain', levels=topolevs)
#ax.contour( rtk.T )
"""
ax=axs[1]
#ax.contour( Terr.suba[k].T )
ax.contourf( rtk.T , cmap='terrain', levels=topolevs )






clng = List.clngth[ipk-1]          # cube cells
c    = Terr.PSW                    # 32 — centre of the 65x65 window
half = clng / 2.0

ax.plot([c, c], [c-half, c+half], lw=3.5, color='red', zorder=5,
        solid_capstyle='butt')
ax.plot([c], [c], 'r+', ms=14, mew=2.5, zorder=6)
ax.text(c+1.5, c+half, f'clngt = {clng*3.34:.0f} km',
        color='red', fontsize=12, va='bottom')


km_ticks(axs[1], x0=Terr.PSW,  y0=Terr.PSW,  step=50)


ax=axs[2]
ax.plot( (Terr.profile(k, ns0=18, ns1=64-18) ))
ax.plot( 18+List.xwedge_list[ipk-1,:] , List.hwedge_list[ipk-1,:] )
xw = 18 + List.xwedge_list[ipk-1, :]
hw = List.hwedge_list[ipk-1, :]
ax.plot(xw, hw)

ax.set_ylim( -500,1000 )
ax.set_xlim(15,47)

h_m, W_km = annotate_wedge(ax, xw, hw)
print(f"wedge: h = {h_m:.0f} m   W = {W_km:.1f} km"
      f"   |  stored mxdis = {List.mxdis[ipk-1]:.0f} m"
      f"   hwdth = {List.hwdth[ipk-1]*GLS:.1f} km")

"""
prof = Terr.profile(k, ns0=18, ns1=64-18)
xp   = np.arange(len(prof))
for idx, m in ((int(np.argmax(prof)), 'v'), (int(np.argmin(prof)), '^')):
    ax.plot(xp[idx], prof[idx], m, ms=11, color='darkorange',
            mec='k', mew=0.8, zorder=8)
"""

ax.set_ylabel("Mean Ridge Topographic deviation (m)" )
km_ticks(axs[2], x0=Terr.PSW,  step=25, axis="x",
         label="cross-ridge distance (km)")



for ax, lab in zip( axs, "cde"):
    panel_label(ax, f"{lab})")

anglx = List.anglx[ipk-1]

north_arrow(axs[0], frac=(0.5, 0.5))
#north_arrow(axs[1], frac=(0.5, 0.5))
rotation_arrow(axs[1], anglx, frac=(0.5, 0.52), sign=1)
north_arrow_rotated(axs[1], anglx , frac=(0.5, 0.5) )
dir_arrow(axs[1],  frac=(0.5, 0.5),label="cross-ridge")
dir_arrow(axs[2], frac=(0.53, 0.021), label="cross-ridge")

#km_ticks(axs[1], x0=Terr.PSW,  y0=Terr.PSW,  step=50)
#km_ticks(axs[2], x0=Terr.PSW,  step=25, axis="x",
#         label="cross-ridge distance (km)")




"""

anglx = List.anglx[ipk-1]

north_arrow(axs[0])
north_arrow(axs[1])
rotation_arrow(axs[2], anglx, frac=(0.5, 0.52), sign=-1)
dir_arrow(axs[2], label="cross-ridge")
dir_arrow(axs[3], frac=(0.55, 0.06), label="cross-ridge")
"""

In [ ]:
print(ipk)

In [ ]:
%matplotlib inline

k=k_smokies # k_smokies #k_sangab

rtk=Terr.rt[k]
rtk=np.where( rtk < -10000., 0., rtk )
print( rtk.shape )



ipk = Terr.ipk[k]
print(List.mxdis[ipk-1], List.anglx[ipk-1], List.clngth[ipk-1],List.NSW,List.PSW)

xedge=List

print( List.hwedge_list[ipk-1,:] )
print( 17+List.xwedge_list[ipk-1,:] )

print( len(oo) )
print( Terr.xspk[k],Terr.yspk[k])
xspk,yspk= Terr.xspk[k],Terr.yspk[k]
fig,axs=plt.subplots( 1,4, figsize=(41,9) )

i0, j0 = int(round(xspk)), int(round(yspk))
maxraw = np.max(dev[3, j0-30:j0+30, i0-30:i0+30])

#topolevs=np.linspace( 0,maxraw,num=51)

topolevs = np.linspace(-1000., 1000., 21)

ax=axs[0]
#ax.contour(Rmp.mxdis[3,:,:] )
ax.contourf(dev[3,:,:] , cmap='terrain', levels=topolevs)
ax.contour(Rmp.mxdis[3,:,:] , colors='black')
ax.set_xlim(xspk-30,xspk+30) # 300,800 )
ax.set_ylim(yspk-30,yspk+30) #2500,3000 )
ax.plot( xspk,yspk , '+'  )

ax=axs[1]
ax.contour( Terr.suba[k].T )
#ax.contour( rtk.T )

ax=axs[2]
#ax.contour( Terr.suba[k].T )
ax.contour( rtk.T )

ax=axs[3]
ax.plot( (Terr.profile(k, ns0=18, ns1=64-18) ))
ax.plot( 18+List.xwedge_list[ipk-1,:] , List.hwedge_list[ipk-1,:] )

ax.set_ylim( -500,1000 )
ax.set_xlim(15,47)





anglx = List.anglx[ipk-1]

north_arrow(axs[0], frac=(0.5, 0.5))
north_arrow(axs[1], frac=(0.5, 0.5))
rotation_arrow(axs[2], anglx, frac=(0.5, 0.52), sign=1)
north_arrow_rotated(axs[2], anglx , frac=(0.5, 0.5) )
dir_arrow(axs[2], label="cross-ridge")
dir_arrow(axs[3], frac=(0.55, 0.06), label="cross-ridge")


"""

anglx = List.anglx[ipk-1]

north_arrow(axs[0])
north_arrow(axs[1])
rotation_arrow(axs[2], anglx, frac=(0.5, 0.52), sign=-1)
dir_arrow(axs[2], label="cross-ridge")
dir_arrow(axs[3], frac=(0.55, 0.06), label="cross-ridge")
"""

In [ ]:
Rmp.uniqid.shape

In [ ]:
fig,axs=plt.subplots( 1,2, figsize=(21,9) )
ax=axs[0]
ax.contour(Rmp.mxdis[3,:,:] )
ax.contour(raw[3,:,:] , levels=51, alpha=0.2)
#ax.set_xlim(1500,2000 )
#ax.set_ylim(2500,3000 )
ax.set_xlim(1650,1750 )
ax.set_ylim(2650,2750 )

ax=axs[1]
#ax.plot( Rmp.uniqid[3,2840,:] )
ax.plot( Rmp.mxdis[3,:,1720] )
#ax.plot( Rmp.uniqid[3,:,530] )
ax.set_xlim(2600,2700 )
#ax.set_ylim(265_000,270_000 )

#print( Rmp.uniqid[3,2840,500:600] )
print( Rmp.uniqid[3,2600:2700,1720] )

#Sierras
rid=265570

#San Gabriel
rid=263235,262968

In [ ]:
fig,axs=plt.subplots( 1,2, figsize=(21,9) )
ax=axs[0]
ax.contour(Rmp.mxdis[3,:,:] )
ax.set_xlim(300,800 )
ax.set_ylim(2500,3000 )
#ax.set_xlim(500,600 )
#ax.set_ylim(2800,2900 )
ax.set_xlim(500,600 )
ax.set_ylim(2700,2800 )

ax=axs[1]
#ax.plot( Rmp.uniqid[3,2840,:] )
ax.plot( Rmp.mxdis[3,:,570] )
#ax.plot( Rmp.uniqid[3,:,530] )
ax.set_xlim(2700,2800 )
#ax.set_ylim(265_000,270_000 )

#print( Rmp.uniqid[3,2840,500:600] )
print( Rmp.uniqid[3,2700:2800,570] )

#Sierras
rid=265570

#San Gabriel
rid=262660 # 263235,262968

In [ ]:
#import numpy as np
from paint_tiles_nc import cube_index_from_lonlat, lonlat_from_cube_index

x0, y0, p0 = cube_index_from_lonlat(np.array([-117.85]), np.array([34.30]), 3000)
x0, y0, p0 = float(x0[0]), float(y0[0]), int(p0[0])
print(f"target: panel {p0}  ix {x0:.0f}  iy {y0:.0f}")

In [ ]:
d = np.hypot(Terr.xs - x0, Terr.ys - y0)
near = np.nonzero((Terr.panel == p0) & (d <= 15))[0]     # ~50 km
print(f"{len(near)} stored windows within 15 cells")

In [ ]:
from rdridgelist import rdridgelist, latest_ridge_list
R = rdridgelist(latest_ridge_list(case_dir), verbose=True)
assert R.npeaks == Terr.npeaks, "TerrXY and Ridge_list are from different runs"

ids   = Terr.ipk[near] - 1                 # 0-based into the ridge list
order = np.argsort(-(R.mxdis[ids] * R.aniso[ids]))   # tall AND well-defined

for n in order[:10]:
    k, i = near[n], ids[n]
    lo, la = lonlat_from_cube_index(np.array([Terr.xs[k]]), np.array([Terr.ys[k]]),
                                    np.array([Terr.panel[k]]), 3000)
    print(f"slot {k:6d} peak {Terr.ipk[k]:7d}  mxdis {R.mxdis[i]:7.1f} m"
          f"  anglx {R.anglx[i]:6.1f}  clngth {R.clngth[i]:6.1f}"
          f"  aniso {R.aniso[i]:5.2f}   {np.degrees(la[0]):5.2f}N {np.degrees(lo[0])-360:7.2f}E")

In [ ]:
Rmp.keys()

In [ ]:
aniso[aniso < 0.] = 0.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import cartopy.crs as ccrs


# ============================================================
# 1. Choose geographical region
# ============================================================

location = 'Colorado' # 'Mogollon'  

location_info = {

    'Colorado': {
        'panel': 4,
        'lonrange': np.asarray([-117., -100.]) + 360.,
        'latrange': np.asarray([30., 41.]),
        'projection': 'LambertConformal'
    },

    'California': {
        'panel': 4,
        'lonrange': np.asarray([-125., -108.]) + 360.,
        'latrange': np.asarray([30., 41.]),
        'projection': 'LambertConformal'
    },

    'Arkansas': {
        'panel': 4,
        'lonrange': np.asarray([-102., -85.]) + 360.,
        'latrange': np.asarray([30., 41.]),
        'projection': 'LambertConformal'
    },

    'Tennessee': {
        'panel': 4,
        'lonrange': np.asarray([-92., -75.]) + 360.,
        'latrange': np.asarray([30., 41.]),
        'projection': 'LambertConformal'
    },

    'Utah': {
        'panel': 4,
        'lonrange': np.asarray([-113., -108.]) + 360.,
        'latrange': np.asarray([36., 41.]),
        'projection': 'LambertConformal'
    },

    'Mogollon': {
        'panel': 4,
        'lonrange': np.asarray([-111., -110.]) + 360.,
        'latrange': np.asarray([34., 35.]),
        'projection': 'LambertConformal'
    },

    'Kaibab': {
        'panel': 4,
        'lonrange': np.asarray([-113., -111.]) + 360.,
        'latrange': np.asarray([36., 37.]),
        'projection': 'LambertConformal'
    },

    'SouthwestUS': {
        'panel': 4,
        'lonrange': np.asarray([-130., -104.]) + 360.,
        'latrange': np.asarray([29., 42.]),
        'projection': 'AlbersEqualArea'
    },

    'CONUS_South': {
        'panel': 4,
        'lonrange': np.asarray([-125., -70.]) + 360.,
        'latrange': np.asarray([24., 45.]),
        'projection': 'AlbersEqualArea'
    },

    'SouthAndesAntarcticPen': {
        'panel': 5,
        'lonrange': np.asarray([260., 300.]),
        'latrange': np.asarray([-75., -50.]),
        'projection': 'Orthographic'
    },

    'McMurdo': {
        'panel': 5,
        'lonrange': np.asarray([120., 230.]),
        'latrange': np.asarray([-87., -50.]),
        'projection': 'Orthographic'
    },

    'Antarctica': {
        'panel': 5,
        'lonrange': np.asarray([0., 360.]),
        'latrange': np.asarray([-90., -50.]),
        'projection': 'Orthographic'
    }
}


if location not in location_info:
    raise ValueError(f"Unknown location: {location}")

cfg = location_info[location]

panel    = cfg['panel']
ip       = panel - 1
lonrange = cfg['lonrange']
latrange = cfg['latrange']

central_longitude = np.average(lonrange)
central_latitude  = np.average(latrange)

In [ ]:
# ============================================================
# 2. Set map projection
# ============================================================

if cfg['projection'] == 'LambertConformal':

    MapProj = ccrs.LambertConformal(
        central_longitude=central_longitude,
        central_latitude=central_latitude
    )

elif cfg['projection'] == 'AlbersEqualArea':

    MapProj = ccrs.AlbersEqualArea(
        central_longitude=central_longitude,
        central_latitude=central_latitude
    )

elif cfg['projection'] == 'Orthographic':

    if location == 'Antarctica':
        central_longitude = 180.
        central_latitude  = -90.

    MapProj = ccrs.Orthographic(
        central_longitude=central_longitude,
        central_latitude=central_latitude
    )

DataProj = ccrs.PlateCarree()


# ============================================================
# 3. Find target cells in region
# ============================================================

in_region = (
    (target_lon >  lonrange[0]) &
    (target_lon <= lonrange[1]) &
    (target_lat >  latrange[0]) &
    (target_lat <= latrange[1])
)

iroo = np.where(in_region)[0]

#iroo=iroo[0:1]
print(f"Number of target cells in region = {len(iroo)}")


# ============================================================
# 4. Find tile subsection needed for plot
# ============================================================

tile_mask = (
    (tlons[ip, :, :] >  lonrange[0]) &
    (tlons[ip, :, :] <= lonrange[1]) &
    (tlats[ip, :, :] >  latrange[0]) &
    (tlats[ip, :, :] <= latrange[1])
)

if not np.any(tile_mask):
    raise ValueError("Requested region does not overlap selected topo tile")

yy_idx, xx_idx = np.where(tile_mask)

xxrange = [xx_idx.min(), xx_idx.max()]
yyrange = [yy_idx.min(), yy_idx.max()]

print("x range =", xxrange)
print("y range =", yyrange)
print("central latitude =", central_latitude)

xslice = slice(xxrange[0], xxrange[1] + 1)
yslice = slice(yyrange[0], yyrange[1] + 1)


# ============================================================
# 5. Plot settings
# ============================================================

topo_levels = np.linspace(-1000., 1000., 21)
cmap = plt.get_cmap('terrain')

# Same mapping for shaded topo and ridgelet coloring
norm = mpl.colors.BoundaryNorm(
    topo_levels,
    ncolors=cmap.N,
    clip=True
)

ridge_linewidth = 1.8


# ============================================================
# 7. Create figure
# ============================================================

x_npanels = 3 
fig = plt.figure(figsize=( x_npanels*12.1 , 12))

#ax1 = fig.add_axes([0.05, 0.12, 0.40, 0.78], projection=MapProj)
#ax2 = fig.add_axes([0.52, 0.12, 0.40, 0.78], projection=MapProj)

x_LL, x_wd=0.03,0.28
ax1 = fig.add_axes([x_LL, 0.12, x_wd, 0.78], projection=MapProj)
x_LL=x_LL+0.33
ax2 = fig.add_axes([x_LL, 0.12, x_wd, 0.78], projection=MapProj)
x_LL=x_LL+0.33
ax3 = fig.add_axes([x_LL, 0.12, x_wd, 0.78], projection=MapProj)

cf1 = plot_ridgelet_panel(
    ax1,
    fill_field=dev,
    plot_ridgelets=False,
    panel_title="Dev topography shading "
)


cf2 = plot_ridgelet_panel(
    ax2,
    fill_field=0.5*dev,
    ridgelet_length_fac=3.3*aniso,
    ridgelet_color_override=None,
    alpha=0.2,
    panel_title=f"Dev topography shading + \nRidgelet color = mxdis (drawn w/ {lngthvar})"
)

cf3 = plot_ridgelet_panel(
    ax3,
    fill_field=0.5*dev,
    ridgelet_length_fac=3.3*aniso, #aniso,
    ridgelet_color_override='black',
    alpha=0.5,
    panel_title=f"Dev topography shading + \nRidgelets - black (drawn w/ {lngthvar} )"
)


# ============================================================
# 8. Shared colorbar
# ============================================================


cax = fig.add_axes([0.99, 0.18, 0.02, 0.62])

sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])

cbar = fig.colorbar(sm, cax=cax)
cbar.set_label('Elevation-like value (m)', fontsize=12)
# If you prefer to be more explicit:
# cbar.set_label('mxdis + target_topo (m)', fontsize=12)


# ============================================================
# 9. Suptitle
# ============================================================

fig.suptitle(
    "3000x3000 topography over-plotted with ridgelets\n"
    f"{TileFile}",
    fontsize=15,
    y=0.97
)

plt.show()

In [ ]:
print( Rmp.uniqid.max() )


In [ ]:
plt.plot( Tile.mxdis[iroo,:].flatten(),'.')
plt.ylim(0,2000)
plt.xlim(1300,1330)

print( Tile.mxdis[iroo,:].flatten()[1320:1330] )
print( Tile.MyObject[iroo,:].flatten()[1320:1330] )


In [ ]:
chid = 160310 # 160223
zoo = np.where( Rmp.uniqid== chid )
poo = np.where( Tile.MyObject == chid )

In [ ]:
zood=np.asarray(zoo)
print( zood.shape )
pood=np.asarray(poo)
print( pood[0] )
print( Tile.MyObject [ pood[0], pood[1] ] )
print( Tile.LnObject [ pood[0], pood[1] ] )
print( zood.shape[1] )
print( np.sum( Tile.LnObject [ pood[0], pood[1] ] ))


In [ ]:
poop = Tile.MyObject[iroo,:]
poop2=poop[ poop > 0] 
print( poop2.min(), poop2.max() )

test1 = np.zeros( poop2.max() - poop2.min() )
test2 = np.zeros( poop2.max() - poop2.min() )
print( test1.shape )
ii=0
#for chid in np.arange( poop2.min(), poop2.max() + 1 ):
for chid in np.arange( 160000,160100 + 1 ):
    zoo = np.where( Rmp.uniqid== chid )
    boo = np.where( Tile.MyObject == chid )
    zood=np.asarray(zoo)
    bood=np.asarray(boo)
    test1[ii]=zood.shape[1]
    test2[ii]= np.sum( Tile.LnObject [ bood[0], bood[1] ] )
    ii=ii+1
    if ii%100==0:
        print(ii)


In [ ]:
npeaks = List.npeaks

cube = RM.count_ids(Rmp.uniqid, npeaks)          # cells per ID on the cube

v = Tile.valid()                             # <-- essential
S = np.bincount(Tile.MyObject[v].astype(int),
                weights=Tile.LnObject[v],
                minlength=npeaks+1)[1:]

d = S - cube
print("ridges disagreeing :", int((d != 0).sum()), "of", npeaks)
print("totals             :", cube.sum(), "vs", S.sum())
print("max |difference|   :", np.abs(d).max())

# your original window, ID k -> index k-1
k = np.arange(160000, 161001)
print(np.column_stack([k, cube[k-1], S[k-1]])[:20])

In [ ]:
plt.plot( test1 - test2)
plt.plot( test2 )


In [ ]:
npeaks = r.npeaks                       # from the ridge list

ids = uniqi.astype(np.int64).ravel()    # truncation, matching Fortran
ids = np.where(ids > 0, ids, 0)         # 0 is the ignore bin
cube_count = np.bincount(ids, minlength=npeaks+1)[1:]   # index -> ipk-1


In [ ]:
r2d=180./np.pi

print( iroo )
print( Tile.MyObject[iroo,:] )
#print( TileMap.mxdis[iroo,:].astype(int) )
#print( TileMap.anglx[iroo,:].astype(int) )
#print( TileMap.angll[iroo,:].astype(int) )
print( Tile.clngt[iroo,:].astype(int) )
#print( r2d*TileMap.latc[iroo,:] )
#print( r2d*TileMap.lonc[iroo,:] )
#print( TileMap.yloid[iroo,:].astype(int) )
#print( TileMap.xloid[iroo,:].astype(int) )

print( Tile.LnObject[iroo,:]  ) #.astype(int) )



In [ ]:
from dataclasses import fields
[f.name for f in fields(Tile)]

In [ ]:
plt.plot( TileMap.aniso[iroo,:].flatten(), '.' )

In [ ]:
print(TileFile)

TileFile0 = '/glade/work/juliob/Topo/NCARTopoJTB-fire//cases/ne30pg3_Sco100_Sfi6_v200_rch0p5/output/Ridge_tile_map.dat'

In [ ]:
r1 = Ti.rdtilemap(TileFile0, read_angll=True)
r2 = Ti.rdtilemap(TileFile,  read_angll=True)
v1, v2 = r1.valid(), r2.valid()
print(r1.LnObject[v1].mean(), r2.LnObject[v2].mean())   # did in-box crest grow?
print(r1.clngt[v1].mean(),    r2.clngt[v2].mean())      # should be identical
print((r1.clngt[v1] > 42).mean())                        # fraction ever truncated

In [ ]:
r2.clngt2.shape

In [ ]:
plt.plot(r2.clngt2,'.')

In [ ]:
plt.scatter( RdgList.clngth, r2.clngt2, s=0.1  )
plt.xlabel( "Ridge List clngth" )
plt.ylabel( "Tile Map clngt2" )


In [ ]:
t=TileMap
r=RdgList
v   = t.valid()
ids = t.MyObject[v].astype(int)            # 1-based ridge IDs
S   = np.bincount(ids, weights=t.LnObject[v], minlength=r.npeaks+1)[1:]

def lis(a):                                 # length_in_square, vectorised
    ar = np.radians(np.asarray(a)); d = np.full(ar.shape, np.nan)
    m = (ar > -np.pi/4) & (ar <= np.pi/4);      d[m] = 1/np.cos(ar[m])
    m = (ar > np.pi/4) & (ar <= 3*np.pi/4);     d[m] = 1/np.cos(ar[m]-np.pi/2)
    m = (ar > 3*np.pi/4) & (ar <= 5*np.pi/4);   d[m] = 1/np.cos(ar[m]-np.pi)
    return d

nsw   = r.NSW
ncl   = np.minimum((r.clngth//2).astype(int), nsw)
drawn = 2*ncl + 1                            # cells in suba before rotation
pred  = drawn / lis(r.anglx)                 # cells after nearest-neighbour rotation

good = r.mxdis > 0
plt.scatter(pred[good], S[good], s=1, alpha=0.2)
plt.plot([0, pred[good].max()], [0, pred[good].max()], 'k-')
plt.xlabel('predicted painted crest cells'); plt.ylabel('sum(LnObject)')

In [ ]:
ratio = np.where(pred > 0, S/np.maximum(pred,1), np.nan)[good]
print("retained fraction: median %.2f  mean %.2f" % (np.nanmedian(ratio), np.nanmean(ratio)))
print("ridges retaining nothing: %.1f%%" % (100*np.nanmean(ratio < 0.05)))

# are the losers the unimportant ones?
plt.scatter(pred[good], S[good], s=2, c=np.log10(r.mxdis[good]), cmap='viridis')
plt.colorbar(label='log10 mxdis')

In [ ]:
m = good & (r.mxdis >= 1.0)          # paintridge2cube skips mxdis < 1
ratio = S[m] / np.maximum(pred[m], 1)
lm = np.log10(r.mxdis[m])

print("corr(log10 mxdis, retained) = %.3f" % np.corrcoef(lm, ratio)[0,1])
for lo, hi in [(1,10),(10,100),(100,1000),(1000,1e9)]:
    s = m & (r.mxdis>=lo) & (r.mxdis<hi)
    if s.sum():
        rr = S[s]/np.maximum(pred[s],1)
        print(f"  mxdis {lo:>6}-{hi:<6}: n={s.sum():6d}  median retained={np.median(rr):.2f}"
              f"  frac lost entirely={np.mean(rr<0.05):.2f}")